# 04 — VGA3d Extension

This notebook extends our 2D understanding to 3D Vector Geometric Algebra (VGA3d). We explore bivectors, trivectors, the relationship to the cross product, and 3D rotors.

## Learning Objectives

- Understand the 8-dimensional VGA3d algebra
- Work with bivectors and trivectors in 3D
- Connect GA operations to classical cross product and triple product
- Construct and apply 3D rotors
- Compute signed volumes

In [ ]:
# Setup
from amsa import Algebra
import numpy as np
import matplotlib.pyplot as plt

alg = Algebra.vga3d()

## 4.1 The VGA3d Basis

VGA3d has 2³ = 8 basis blades organized by grade:

| Grade | Blades | Description |
|-------|--------|-------------|
| 0 | e | Scalar |
| 1 | e1, e2, e3 | Vectors |
| 2 | e12, e13, e23 | Bivectors (oriented areas) |
| 3 | e123 | Trivector (oriented volume) |

The bivectors correspond to the three coordinate planes: e12 (xy), e13 (xz), e23 (yz).

In [ ]:
# Explore the VGA3d basis
print("VGA3d has", alg.spec.blade_count, "basis blades:")
grades = alg.spec.grades_of_blades()
for blade_index in range(alg.spec.blade_count):
    name = alg.spec.blade_name(blade_index)
    grade = grades[blade_index]
    print(f"  Blade {blade_index}: '{name}' (grade {grade})")

## 4.2 Vectors in 3D

Creating 3D vectors works just like 2D, but with three components.

In [ ]:
# 3D vectors
v1 = alg.vector([1.0, 0.0, 0.0])
v2 = alg.vector([0.0, 1.0, 0.0])
v3 = alg.vector([0.0, 0.0, 1.0])

print("e1:", v1.values)
print("e2:", v2.values)
print("e3:", v3.values)

# Arbitrary vector
v = alg.vector([2.0, 3.0, 4.0])
print("\nVector [2, 3, 4]:", v.values)
print("Components: e1=", v.component("e1"), ", e2=", v.component("e2"), ", e3=", v.component("e3"))

## 4.3 Bivectors (Grade 2)

Bivectors in 3D represent oriented plane elements. There are three basis bivectors:

- e12 → xy plane
- e13 → xz plane  
- e23 → yz plane

They are dual to vectors in 3D: each bivector corresponds to a normal vector.

In [ ]:
# Bivector construction
b12 = alg.multivector({"e12": 1.0})  # xy plane
b13 = alg.multivector({"e13": 1.0})  # xz plane
b23 = alg.multivector({"e23": 1.0})  # yz plane

print("e12 (xy plane):", b12.values)
print("e13 (xz plane):", b13.values)
print("e23 (yz plane):", b23.values)

# Bivector from two vectors
u = alg.vector([1.0, 0.0, 0.0])
v = alg.vector([0.0, 1.0, 0.0])
b = u ^ v  # outer product

print("\nu ^ v (e1 ^ e2):", b.values)

## 4.4 Cross Product Relationship

In 3D, the cross product relates to both the outer product and the dual:

$$u \times v = \text{dual}(u \wedge v)$$

Let's verify this.

In [ ]:
# Compare cross product with GA
u = alg.vector([1.0, 2.0, 3.0])
v = alg.vector([4.0, 5.0, 6.0])

# Classical cross product
cross = np.cross(u.grade(1).values, v.grade(1).values)

# GA: outer product then dual
outer = u ^ v
dual_bivector = outer.dual()

print("Cross product (NumPy):", cross)
print("GA dual of outer product:", dual_bivector.grade(1).values)
print("\nResults match:", np.allclose(cross, dual_bivector.grade(1).values))

## 4.5 Trivectors and Volume (Grade 3)

The highest-grade element in VGA3d is the trivector e123 — representing oriented volume. The outer product of three vectors produces a trivector.

In [ ]:
# Trivector from three vectors
u = alg.vector([1.0, 0.0, 0.0])
v = alg.vector([0.0, 2.0, 0.0])
w = alg.vector([0.0, 0.0, 3.0])

trivector = u ^ v ^ w

print("u ^ v ^ w:", trivector.values)
print("e123 component:", trivector.component("e123"))
print("\nVolume:", 1.0 * 2.0 * 3.0)  # 1*2*3 = 6

In [ ]:
# The triple product relates to determinant
u = alg.vector([1.0, 0.0, 0.0])
v = alg.vector([1.0, 1.0, 0.0])
w = alg.vector([1.0, 1.0, 1.0])

tri = u ^ v ^ w
volume = tri.component("e123")

matrix = np.array([u.grade(1).values, v.grade(1).values, w.grade(1).values])
det = np.linalg.det(matrix)

print("GA trivector component:", volume)
print("Determinant (classical):", det)
print("\nNote: they match because trivector = determinant * e123")

## 4.6 Signed Volume Visualization

The sign of the trivector indicates orientation (right-handed vs left-handed).

In [ ]:
fig = plt.figure(figsize=(12, 5))

# Right-handed orientation
ax1 = fig.add_subplot(121, projection='3d')
u_vec = np.array([1, 0, 0])
v_vec = np.array([0, 1, 0])
w_vec = np.array([0, 0, 1])

ax1.quiver(0, 0, 0, *u_vec, color='blue', linewidth=2, label='u')
ax1.quiver(0, 0, 0, *v_vec, color='red', linewidth=2, label='v')
ax1.quiver(0, 0, 0, *w_vec, color='green', linewidth=2, label='w')

# Draw the parallelepiped
corners = np.array([[0,0,0], u_vec, v_vec, w_vec, u_vec+v_vec, u_vec+w_vec, v_vec+w_vec, u_vec+v_vec+w_vec])
ax1.scatter(*corners.T, alpha=0.5)

tri = alg.vector(u_vec) ^ alg.vector(v_vec) ^ alg.vector(w_vec)
ax1.set_title(f'Right-handed orientation\nVolume = {tri.component("e123"):.1f}', fontsize=11)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

# Left-handed orientation
ax2 = fig.add_subplot(122, projection='3d')
u_vec = np.array([1, 0, 0])
v_vec = np.array([0, 1, 0])
w_vec = np.array([0, 0, -1])  # reversed

ax2.quiver(0, 0, 0, *u_vec, color='blue', linewidth=2, label='u')
ax2.quiver(0, 0, 0, *v_vec, color='red', linewidth=2, label='v')
ax2.quiver(0, 0, 0, *w_vec, color='green', linewidth=2, label='w')

tri2 = alg.vector(u_vec) ^ alg.vector(v_vec) ^ alg.vector(w_vec)
ax2.set_title(f'Left-handed orientation\nVolume = {tri2.component("e123"):.1f}', fontsize=11)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

plt.tight_layout()
plt.show()

## 4.7 3D Rotors

Rotors in 3D are more interesting than 2D: they can rotate around any axis (specified by a bivector). A 3D rotor is:

$$R = \cos(\theta/2) - B \sin(\theta/2)$$

where $B$ is a unit bivector defining the rotation plane.

In [ ]:
# Create a rotor around the z-axis (e12 plane)
theta = np.pi / 4  # 45 degrees

# Rotation in the xy-plane (around z-axis)
rotor_z = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

print("Rotor around z-axis:", rotor_z.values)

# Apply to a vector in the xy-plane
v = alg.vector([1.0, 0.0, 0.0])
v_rotated = rotor_z.sandwich(v)

print("Original:", v.grade(1).values)
print("Rotated:", v_rotated.grade(1).values)
print("\nExpected (45° around z): [cos(45°), sin(45°), 0] =", [np.cos(theta), np.sin(theta), 0])

In [ ]:
# Rotation around an arbitrary axis
# Let's rotate around the axis defined by (1, 1, 0) direction
# This corresponds to the bivector e12 + e13 (not normalized, but for illustration)

# First create a normalized bivector for the rotation plane
axis = alg.vector([1.0, 1.0, 0.0]).normalized()

# Get the bivector normal to this axis (use dual)
# For axis in xy-plane, the bivector is e12
B = alg.multivector({"e12": 1.0})  # xy-plane bivector

# Create rotor
theta = np.pi / 3  # 60 degrees
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Test on various vectors
test_vectors = [
    alg.vector([1.0, 0.0, 0.0]),
    alg.vector([0.0, 1.0, 0.0]),
    alg.vector([0.0, 0.0, 1.0])
]

print("Rotation around z-axis by 60°:")
for tv in test_vectors:
    rotated = rotor.sandwich(tv).grade(1)
    print(f"  {tv.grade(1).values} -> {rotated.values}")

## 4.8 Visualizing 3D Rotation

Let's visualize a 3D rotor rotating points on a sphere.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create points on a unit sphere
n_points = 20
theta_vals = np.linspace(0, 2*np.pi, n_points, endpoint=False)
phi_vals = np.linspace(0, np.pi, n_points//2)

points_orig = []
for phi in phi_vals:
    for theta in theta_vals:
        x = np.sin(phi) * np.cos(theta)
        y = np.sin(phi) * np.sin(theta)
        z = np.cos(phi)
        points_orig.append([x, y, z])

points_orig = np.array(points_orig)

# Create rotor for 45-degree rotation around z-axis
theta_rot = np.pi / 4
rotor = alg.multivector({
    "e": np.cos(theta_rot / 2),
    "e12": -np.sin(theta_rot / 2)
}).normalized()

# Rotate all points
points_rot = []
for p in points_orig:
    mv = alg.vector(p)
    rotated = rotor.sandwich(mv).grade(1).values
    points_rot.append(rotated)
points_rot = np.array(points_rot)

# Plot
fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points_orig[:, 0], points_orig[:, 1], points_orig[:, 2], c='blue', alpha=0.5, s=20)
ax1.set_title('Original Sphere', fontsize=11)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(points_rot[:, 0], points_rot[:, 1], points_rot[:, 2], c='red', alpha=0.5, s=20)
ax2.set_title('After 45° Rotation (z-axis)', fontsize=11)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

plt.tight_layout()
plt.show()

## 4.9 Summary

We covered:

- **VGA3d**: 8 basis blades (scalar + 3 vectors + 3 bivectors + trivector)
- **Bivectors**: oriented plane elements, dual to vectors in 3D
- **Cross product**: $u \times v = \text{dual}(u \wedge v)$
- **Trivectors**: oriented volume, triple product relationship
- **3D rotors**: rotate around any bivector plane
- **Volume**: $u \wedge v \wedge w$ gives signed volume

This completes the VGA fundamentals! The next section (02_projective) covers PGA — where the real robotics magic happens.

## Exercises

### ⭐ Easy

**4.1** Create vectors u = [1, 0, 0], v = [0, 1, 0], w = [0, 0, 1]. Compute their outer product and verify the trivector component equals 1.

In [ ]:
# Your turn: ⭐ Exercise 4.1
u = alg.vector([1.0, 0.0, 0.0])
v = alg.vector([0.0, 1.0, 0.0])
w = alg.vector([0.0, 0.0, 1.0])
# TODO: Compute u ^ v ^ w and verify component
raise NotImplementedError("Implement exercise 4.1")

### ⭐⭐ Medium

**4.2** Write code to verify that $u \times v = \text{dual}(u \wedge v)$ for three different pairs of vectors.

In [ ]:
# Your turn: ⭐⭐ Exercise 4.2
test_pairs = [
    (alg.vector([1, 2, 3]), alg.vector([4, 5, 6])),
    (alg.vector([1, 0, 0]), alg.vector([0, 1, 0])),
    (alg.vector([2, -1, 1]), alg.vector([1, 1, -1]))
]
# TODO: Verify cross = dual(outer) for each pair
raise NotImplementedError("Implement exercise 4.2")

### ⭐⭐⭐ Challenge

**4.3** Create a function `rotor_3d(axis_vector, angle)` that takes a 3D axis vector and angle, and returns the corresponding rotor. Test it by rotating [1, 0, 0] around [0, 1, 0] by 90°.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 4.3
def rotor_3d(axis_vector, angle):
    """Create a 3D rotor for rotation around given axis."""
    # TODO: Implement
    raise NotImplementedError("Implement rotor_3d function")

# Test: rotate around y-axis by 90 degrees
axis = alg.vector([0.0, 1.0, 0.0])
R = rotor_3d(axis, np.pi/2)
v = alg.vector([1.0, 0.0, 0.0])
result = R.sandwich(v).grade(1)
print("Original:", v.grade(1).values)
print("Rotated around y-axis:", result.values)
print("Expected: [0, 0, -1]")

## Attribution

This notebook draws on:

- **Linear and Geometric Algebra** — Alan Macdonald
  https://www.faculty.luther.edu/~macdonal/laga/
- **Geometric Algebra for Computer Science** — Dorst, Lewiner, et al.
  https://geometricalgebra.org/
- **Geometric Algebra for Computer Graphics** — John Vince, Springer 2008
  https://link.springer.com/book/10.1007/978-1-84628-997-2
- **Geometric Algebra for Physicists** — Hollmeier, Heidelberg University
  https://physi.uni-heidelberg.de/~vhollmeier/ga4p/materials/ga_lecture_notes.pdf